# Step 0 of the PW: data collection



In [1]:
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter
from sqlalchemy import and_, case, distinct, func, literal, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # import the needed tables for the extractions
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS206_PERSON,
    TLS207_PERS_APPLN,
    TLS224_APPLN_CPC,
    TLS230_APPLN_TECHN_FIELD,
    TLS901_TECHN_FIELD_IPC,
)

In [2]:
# ---------------------------------------------------------------- environment
ENV = "TEST"                 # develop on TEST, switch to "PROD" for the final numbers
OUTPUT_DIR = Path("outputs/step0") # directory for the extraction and hand-labeling outputs

# ---------------------------------------------------------------- sector definition
SECTOR_NAME = "Underwater drones"

SECTOR_IPC_PREFIXES = (      # IPC codes for global macro-categories
    "B63G8/00",              # Submarines and underwater vessels
    "B63G8/08",              # Propulsion specifically for submarines
    "B63G8/14",              # Control of attitude or depth for underwater vessels
    "G01S15/89",             # Sonar systems for mapping or imaging
    "H04B11/00",             # Acoustic communication
    "G01C13/00",             # Surveying specially adapted to open water


)

SECTOR_CPC_PREFIXES = (
    "B63G2008/002",  # Unmanned underwater vessels
    "B63G2008/004",  # Autonomously operating underwater vessels (AUV)
    "B63G2008/005",  # Remotely controlled underwater vessels (ROV)
    "B63G2008/007",  # Tethered underwater vessels
    "B63G2008/008",  # Docking stations for unmanned underwater vessels
)

SECTOR_KEYWORDS = (          # Searched in titles and abstracts
    "underwater drone",
    "subsea drone",
    "underwater vehicle",
    "unmanned underwater vehicle",
    "uncrewed underwater vehicle",
    "autonomous underwater vehicle",
    "remotely operated vehicle",
    "remotely operated underwater",
    "unmanned underwater",
    "autonomous underwater",
    "uncrewed underwater",
    "unmanned submersible",
    "autonomous submersible",
    "uncrewed submersible",
    "underwater glider",
    "subsea robot",
    "underwater robot",
    "submarine robot",
    "subsea resident vehicle",
    "benthic vehicle",
    "hovering autonomous underwater",
    "underwater crawler",
    "seabed crawler",
    "biomimetic underwater robot",
    "robotic fish",
    "swarming underwater",
    "multi-AUV",
)

YEAR_MIN, YEAR_MAX = 2000, 2025

# Which year to use (see Section 3). Three columns of tls201_appln can date a patent:
FAMILY_TIME_COLUMN = "earliest_filing_year"   # family level: the earliest one in the family (PW rule)
APPLN_TIME_COLUMN = "appln_filing_year"       # application level: the application's own filing year

# Last years that are still incomplete in PATSTAT, for each choice of year
TRUNCATED_YEARS = {"appln_filing_year": 3, "earliest_filing_year": 3, "earliest_publn_year": 1}

# ---------------------------------------------------------------- counting rules
IPR_TYPE = "PI"                                  # patents of invention only (excludes utility models "UM")
IP5_OFFICES = ("EP", "US", "CN", "JP", "KR")     # the five largest patent offices (Section 9)
TOP_N = 8                                        


LAST_COMPLETE_YEAR = YEAR_MAX - max(TRUNCATED_YEARS[FAMILY_TIME_COLUMN], TRUNCATED_YEARS[APPLN_TIME_COLUMN])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

patstat = PatstatClient(env=ENV)
db = patstat.orm()
print(f"Sector: {SECTOR_NAME} | env: {ENV} | years {YEAR_MIN}-{YEAR_MAX} "
      f"| family year: {FAMILY_TIME_COLUMN} | application year: {APPLN_TIME_COLUMN} "
      f"| complete years up to {LAST_COMPLETE_YEAR}")

Sector: Underwater drones | env: TEST | years 2000-2025 | family year: earliest_filing_year | application year: appln_filing_year | complete years up to 2022


In [3]:
from sqlalchemy import or_, func, and_
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS224_APPLN_CPC,
    TLS209_APPLN_IPC
)

# ---------------------------------------------------------------- 2. Query Builders 
def keyword_condition(terms):
    """OR-condition: any term appears in the title or abstract (case-insensitive)."""
    title = func.lower(TLS202_APPLN_TITLE.appln_title)
    abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)
    return or_(*(col.like(f"%{t.lower()}%") for t in terms for col in (title, abstract)))

def symbol_prefix_condition(symbol_column, codes):
    """OR-condition: the blank-stripped classification symbol starts with one of the codes."""
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))

# ---------------------------------------------------------------- 3 & 4. Extraction Queries & Union
def sector_family_ids(cpc_prefixes=SECTOR_CPC_PREFIXES, ipc_prefixes=SECTOR_IPC_PREFIXES, keywords=SECTOR_KEYWORDS):
    """returns DOCDB_FAMILY_ID belonging to the sector given by the union of CPC, IPC e Keywords."""
    family_id = TLS201_APPLN.docdb_family_id.label("docdb_family_id")
    
    # 1. Query by CPC
    by_cpc = (
        db.query(family_id)
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, cpc_prefixes))
    )
    
    # 2. Query by IPC
    by_ipc = (
        db.query(family_id)
        .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
        .filter(symbol_prefix_condition(TLS209_APPLN_IPC.ipc_class_symbol, ipc_prefixes))
    )
    
    # 3. Query by Keywords
    by_keywords = (
        db.query(family_id)
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(keyword_condition(keywords))
    )
    
    # UNION removes the duplicates 
    return by_cpc.union(by_ipc, by_keywords).distinct()


sector_ids = sector_family_ids()


q_drone = (
    db.query(
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_id,
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.earliest_filing_year,
        TLS202_APPLN_TITLE.appln_title,
        TLS203_APPLN_ABSTR.appln_abstract,
        TLS203_APPLN_ABSTR.appln_abstract_lg
    )
    .filter(TLS201_APPLN.docdb_family_id.in_(sector_ids.subquery().select()))
    .filter(TLS203_APPLN_ABSTR.appln_abstract_lg == "en")
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
)

df_drone = patstat.df(q_drone)

#Hand-Labeling (Ground Truth)
sample_for_labeling = df_drone.drop_duplicates(subset=["docdb_family_id"]).sample(n=100, random_state=42)
sample_for_labeling.to_csv(OUTPUT_DIR / "step0_hand_labeling.csv", index=False)

sector_ids_df = patstat.df(sector_ids)

sector_ids_df = sector_ids_df.drop_duplicates(
    subset=["docdb_family_id"]
).reset_index(drop=True)

sector_ids_df.to_csv(
    OUTPUT_DIR / "step0_sector_family_ids.csv",
    index=False
)

# Patent SBERT

In [18]:
%pip install -q rank-bm25 sentence-transformers umap-learn

Note: you may need to restart the kernel to use updated packages.


In [4]:
import re
from epo.tipdata.epab import EPABClient
epab = EPABClient(env="PROD") 

def strip_tags(text):
    """Remove the XML tags and collapse the spaces"""
    return re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", str(text))).strip()

def get_claims(publication_number):
    """cleaning claims of an european patent"""
    q = epab.query_publication(number=publication_number)
    res = q.get_results("publication.number, claims", output_type="list", limit=1)
    if res and "claims" in res[0]:
        
        claims_text = strip_tags(res[0]["claims"][0].get("text", ""))
        return claims_text
    return "No claims found"

In [5]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer


# 1. Load the manually labelled dataset

pool = pd.read_csv("outputs/step0/step0_hand_labeling_labeled.csv")

required_columns = [
    "appln_abstract_lg",
    "docdb_family_id",
    "appln_title",
    "appln_abstract",
    "is_relevant"
]

missing_columns = [col for col in required_columns if col not in pool.columns]

if missing_columns:
    raise ValueError(f"Missing columns in the dataset: {missing_columns}")

pool = pool[pool["appln_abstract_lg"] == "en"].reset_index(drop=True)

if pool.empty:
    raise ValueError("No English abstracts found in the dataset.")

if pool["docdb_family_id"].duplicated().any():
    raise ValueError("The dataset contains multiple rows for the same DOCDB family.")

truth = pd.to_numeric(pool["is_relevant"], errors="raise").to_numpy()

if not np.isin(truth, [0, 1]).all():
    raise ValueError("is_relevant must contain only 0 and 1.")

truth = truth.astype(bool)

pool["full_text"] = (
    pool["appln_title"].fillna("").astype(str) + " " +
    pool["appln_abstract"].fillna("").astype(str)
)

pool_texts = pool["appln_abstract"].fillna("").astype(str).tolist()


# 2. Patent-specific stop words

PATENT_STOP_WORDS = {
    "said", "wherein", "whereby", "thereof", "therein", "comprising",
    "comprises", "comprise", "including", "includes", "plurality",
    "least", "configured", "provided", "providing", "present",
    "invention", "disclosed", "disclosure", "embodiment",
    "embodiments", "method", "methods", "device", "devices",
    "apparatus", "system", "systems", "unit", "first", "second",
    "third", "one", "having", "based",
}


# 3. Keyword-based retrieval

if not SECTOR_KEYWORDS:
    raise ValueError("SECTOR_KEYWORDS cannot be empty.")

pattern = "|".join(re.escape(k) for k in SECTOR_KEYWORDS)

by_keywords = (
    pool["full_text"]
    .str.lower()
    .str.contains(pattern, na=False)
    .to_numpy()
)


# 4. Focal patents

FOCAL_ROWS = [56, 52, 27, 37, 59] #Inserted manually

if len(FOCAL_ROWS) != 5 or len(set(FOCAL_ROWS)) != 5:
    raise ValueError("Exactly five distinct focal patents must be selected.")

if min(FOCAL_ROWS) < 0 or max(FOCAL_ROWS) >= len(pool):
    raise ValueError("One or more FOCAL_ROWS are out of range.")

if not truth[FOCAL_ROWS].all():
    raise ValueError("All focal patents must be labelled as relevant.")


# 5. Technology description (Tutorial 0 approach)

UNDERWATER_DRONES_DESCRIPTION = """
Technology concerning unmanned and uncrewed underwater vehicles,
including autonomous underwater vehicles (AUVs) and remotely operated
vehicles (ROVs). These systems are designed to operate underwater
without a human operator on board and may perform autonomous or
remotely controlled navigation, exploration, inspection, monitoring,
mapping and maintenance activities.

The technology includes underwater propulsion, guidance and control,
navigation and positioning systems, obstacle avoidance, underwater
communication, sensors, robotic manipulation and docking systems.
Applications include subsea infrastructure inspection, oceanographic
research, offshore operations, seabed exploration and underwater
intervention.
"""

# 6. TF-IDF and Patent SBERT encoding

pool_tfidf = TfidfVectorizer(
    stop_words=list(ENGLISH_STOP_WORDS | PATENT_STOP_WORDS),
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.5,
    sublinear_tf=True
).fit_transform(pool_texts)

patent_sbert = SentenceTransformer("graziasveva93/sbert-adapt-ub")

pool_patent = patent_sbert.encode(
    pool_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

# 7. Semantic similarity with the technology description (Tutorial 0)

description_embedding = patent_sbert.encode(
    UNDERWATER_DRONES_DESCRIPTION,
    convert_to_numpy=True,
    normalize_embeddings=True
)

description_scores = pool_patent @ description_embedding

# 8. Cosine similarity with focal patents

def focal_score(X):
    sims = cosine_similarity(X, X[FOCAL_ROWS])

    for i, focal_idx in enumerate(FOCAL_ROWS):
        sims[focal_idx, i] = -np.inf

    return sims.max(axis=1)


test_mask = np.ones(len(pool), dtype=bool)
test_mask[FOCAL_ROWS] = False

N_SELECTED = int(by_keywords[test_mask].sum())


def top_n(score, n=N_SELECTED):
    selected = np.zeros(len(score), dtype=bool)

    score_copy = score.copy()
    score_copy[FOCAL_ROWS] = -np.inf

    best_indices = np.argsort(-score_copy)[:n]
    selected[best_indices] = True

    return selected


scores = {
    "TF-IDF": focal_score(pool_tfidf),
    "Patent SBERT-adapt-ub": focal_score(pool_patent)
}

by_tfidf = top_n(scores["TF-IDF"])
by_sbert = top_n(scores["Patent SBERT-adapt-ub"])
by_sbert_description = top_n(description_scores)
# 9. Evaluation

def evaluate_query(selected, truth=truth, mask=test_mask):

    sel_test = selected[mask]
    truth_test = truth[mask]

    tp = (sel_test & truth_test).sum()

    precision = tp / sel_test.sum() if sel_test.sum() else 0.0
    recall = tp / truth_test.sum() if truth_test.sum() else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    return {
        "selected": int(sel_test.sum()),
        "precision": precision,
        "recall": recall,
        "F1": f1
    }



queries = {
    "A · keywords": by_keywords,
    "B · focal similarity, TF-IDF": by_tfidf,
    "B · focal similarity, Patent SBERT": by_sbert,
    "C · description similarity, Patent SBERT": by_sbert_description,
    "A or B (Patent SBERT)": by_keywords | by_sbert,
    "A and B (Patent SBERT)": by_keywords & by_sbert,
}

evaluation_table = pd.DataFrame(
    {name: evaluate_query(sel) for name, sel in queries.items()}
).T.round(3)

print("TECHNICAL APPENDIX: RETRIEVAL EVALUATION ")
print(evaluation_table)


# 10. Patents retrieved by SBERT but not by keywords

only_semantic = by_sbert & ~by_keywords & test_mask

pd.set_option("display.max_colwidth", None)

print(
    f"\n{only_semantic.sum()} PATENTS RETRIEVED BY SBERT "
    "BUT NOT BY KEYWORDS"
)

print(
    pool.loc[
        only_semantic,
        ["appln_title", "appln_abstract"]
    ]
)

# 11. Patents retrieved by SBERT description but not by keywords

only_description = by_sbert_description & ~by_keywords & test_mask

print(
    f"\n{only_description.sum()} PATENTS RETRIEVED BY "
    "SBERT DESCRIPTION BUT NOT BY KEYWORDS"
)

print(
    pool.loc[
        only_description,
        ["appln_title", "appln_abstract"]
    ]
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

TECHNICAL APPENDIX: RETRIEVAL EVALUATION 
                                          selected  precision  recall     F1
A · keywords                                  34.0      0.794   0.964  0.871
B · focal similarity, TF-IDF                  34.0      0.412   0.500  0.452
B · focal similarity, Patent SBERT            34.0      0.294   0.357  0.323
C · description similarity, Patent SBERT      34.0      0.324   0.393  0.355
A or B (Patent SBERT)                         57.0      0.491   1.000  0.659
A and B (Patent SBERT)                        11.0      0.818   0.321  0.462

23 PATENTS RETRIEVED BY SBERT BUT NOT BY KEYWORDS
                                                                                                       appln_title  \
9                                                     Marine environment observation device based on ocean station   
10                                                                                Subsea operations support system   
15  Marine geo

In [27]:
display(
    pool.iloc[FOCAL_ROWS][
        ["docdb_family_id", "appln_title", "is_relevant"]
    ]
)

,docdb_family_id,appln_title,is_relevant
56,16642829,Remotely operated vehicle,1
52,68150103,Reinforcement learning-based AUV behavior planning and motion control method,1
27,44785279,UNMANNED UNDERWATER VEHICLE AND METHOD FOR OPERATING AN UNMANNED UNDERWATER VEHICLE,1
37,45437685,Robotic submersible cleaning system,1
59,23447375,"Method and apparatus for cleaning, viewing and documenting the condition of weldments on offshore platforms",1




# Step 0 – Patent Retrieval and Preliminary Validation

## 1. Sector identification and patent retrieval

The first objective of Step 0 was to identify patent families potentially belonging to the Underwater Drones sector. Since no single retrieval criterion is sufficient to capture all relevant patents, we adopted a multi-source retrieval strategy combining three independent approaches:

- **CPC classification:** patents are retrieved when at least one CPC classification symbol starts with one of the selected sector-specific CPC prefixes.
- **IPC classification:** the same prefix-based approach is applied to the selected IPC codes.
- **Keyword search:** patents are retrieved when at least one of the predefined sector-related keywords appears in either the patent title or abstract.

The two helper functions, `keyword_condition()` and `symbol_prefix_condition()`, implement these filters. The keyword search is case-insensitive and uses SQL LIKE conditions to identify occurrences of the selected terms. The classification search removes spaces from the classification symbols before checking whether they start with one of the selected prefixes.

The three retrieval queries are executed independently within the PATSTAT database:

1. The CPC query joins `TLS201_APPLN` with `TLS224_APPLN_CPC`.
2. The IPC query joins `TLS201_APPLN` with `TLS209_APPLN_IPC`.
3. The keyword query joins the application table with `TLS202_APPLN_TITLE` and `TLS203_APPLN_ABSTR`, which contain patent titles and abstracts.

The resulting sets of patent families are then combined using SQL UNION. This operation creates a single pool of candidate families while removing duplicates across the three retrieval strategies.

The use of the union is intentional: a patent family is considered a potential sector candidate if it satisfies at least one of the three retrieval criteria. This increases the coverage of the initial search, since a relevant patent may be identified through its technical classification even if its title and abstract do not contain the selected keywords, or vice versa.

After retrieving the family identifiers, we extract the corresponding application-level information from PATSTAT, including the DOCDB family identifier, application identifier, application authority, earliest filing year, title, abstract and abstract language.

Only English abstracts are retained for the subsequent analysis, ensuring consistency in the text-processing and NLP procedures.

Finally, we construct a manually labelled sample of 100 distinct DOCDB patent families. The sampling procedure uses `drop_duplicates()` to retain one record per family and `sample(n=100, random_state=42)` to select a reproducible random sample.

The resulting dataset is exported to CSV and manually annotated through the binary variable `is_relevant`, where:

- `1` identifies patents considered relevant to the Underwater Drones sector;
- `0` identifies patents considered not relevant to the sector.

The manual labelling identified **33 relevant patent families out of the 100 sampled families**. These annotations provide the ground truth used in the subsequent retrieval evaluation.

## 2. NLP-based retrieval validation

Once the manual labelling was completed, we conducted a preliminary experiment to compare the performance of different patent retrieval approaches.

The purpose was to assess whether text-based similarity methods could complement the initial keyword search and potentially identify relevant patents that the keyword strategy had missed.

The experiment uses the manually labelled dataset as the evaluation pool. Before processing, the code checks that all required columns are available, retains English abstracts, verifies that each DOCDB family appears only once and ensures that the relevance labels contain only binary values.

Two different text representations are then constructed:

- **Title and abstract:** concatenated into `full_text` and used for the keyword-based retrieval strategy.
- **Abstract only:** used as input for both TF-IDF and Patent SBERT.

### 2.1 Keyword-based retrieval

The first approach reproduces the keyword search on the manually labelled dataset.

All predefined sector keywords are combined into a regular expression, with `re.escape()` ensuring that special characters are interpreted literally. The resulting pattern is applied to the lowercase concatenation of patent titles and abstracts.

A patent is retrieved if at least one keyword occurs in its text.

This approach represents the baseline against which the alternative NLP methods are evaluated.

### 2.2 Focal patents and similarity-based retrieval

The second approach relies on five manually selected focal patents, chosen among those labelled as relevant:

| DOCDB family ID | Patent title |
|---:|---|
| 56 | Remotely operated vehicle |
| 52 | Reinforcement learning-based AUV behavior planning and motion control method |
| 27 | Unmanned underwater vehicle and method for operating an unmanned underwater vehicle |
| 37 | Robotic submersible cleaning system |
| 59 | Method and apparatus for cleaning, viewing and documenting the condition of weldments on offshore platforms |

These patents represent different aspects of the sector, including remotely operated vehicles, autonomous underwater vehicles, motion control and robotic inspection or cleaning applications.

They are used as semantic reference points to identify other patents whose abstracts are similar to those of the selected examples.

Two alternative text-representation methods are tested:

#### TF-IDF

The first method applies `TfidfVectorizer` to the patent abstracts. It represents each document through the relative importance of its words and word combinations.

The vectorizer uses unigrams and bigrams, removes standard English stop words and additional patent-specific stop words, and applies frequency thresholds to reduce the influence of terms that are either too rare or too common.

#### Patent SBERT

The second method uses the pretrained model `graziasveva93/sbert-adapt-ub`, specifically designed for patent-related semantic representations.

Unlike TF-IDF, which primarily captures lexical information, SBERT generates dense vector embeddings intended to represent the semantic content of each abstract.

Both approaches calculate cosine similarity between every patent in the pool and the five focal patents. For each document, the highest similarity score across the five reference patents is retained.

The focal patents themselves are excluded from the retrieval results, preventing the reference documents from artificially improving the evaluation.

To make the comparison controlled, both TF-IDF and SBERT retrieve the same number of patents as the keyword baseline: **34 patents in the evaluation subset**.

### 2.3 Description-based semantic retrieval with Patent SBERT

In addition to the focal-patent approach, we conducted a second SBERT experiment based on a textual description of the Underwater Drones sector.

Instead of using individual patents as semantic reference points, this approach uses a general description intended to capture the main characteristics of the sector.

The description covers:

- Unmanned and uncrewed underwater vehicles, including AUVs and ROVs.
- Autonomous and remotely controlled underwater navigation and operation.
- Exploration, inspection, monitoring, mapping and maintenance activities.
- Propulsion, guidance, control, navigation and positioning systems.
- Obstacle avoidance, underwater communication, sensors, robotic manipulation and docking systems.
- Applications in subsea infrastructure inspection, oceanographic research, offshore operations, seabed exploration and underwater intervention.

The description is encoded using the same pretrained model, `graziasveva93/sbert-adapt-ub`, used in the focal-patent experiment.

The resulting description embedding is compared with the previously generated patent abstract embeddings through cosine similarity. Since the embeddings are normalized, cosine similarity can be calculated using their dot product.

Each patent is therefore ranked according to its semantic similarity to the overall sector description.

As in the previous experiments, the approach retrieves the same number of patents as the keyword baseline: **34 patents**. This makes it possible to compare the results using the same evaluation procedure.

This experiment follows the semantic retrieval principle presented in Tutorial 0, where a textual technology description is used as a query against patent abstracts. However, the description-based experiment is an additional evaluation alongside the focal-patent approach.

## 3. Retrieval evaluation

The performance of the different approaches is assessed using three standard information retrieval metrics:

- **Precision:** the proportion of retrieved patents that are actually relevant.
- **Recall:** the proportion of all relevant patents in the evaluation sample that are successfully retrieved.
- **F1-score:** the harmonic mean of precision and recall, providing a combined measure of retrieval performance.

The five focal patents are excluded from the evaluation because they are used as reference documents. Consequently, the metrics are calculated on the remaining **95 patent families**, among which 28 are labelled as relevant.

The evaluation results are reported below.

| Retrieval strategy | Selected | Precision | Recall | F1 |
|:---|---:|---:|---:|---:|
| A – Keywords | 34 | 0.794 | 0.964 | 0.871 |
| B – TF-IDF focal similarity | 34 | 0.412 | 0.500 | 0.452 |
| B – Patent SBERT focal similarity | 34 | 0.294 | 0.357 | 0.323 |
| C – Patent SBERT description similarity | 34 | 0.324 | 0.393 | 0.355 |
| A OR B – Keywords + focal SBERT | 57 | 0.491 | 1.000 | 0.659 |
| A AND B – Keywords + focal SBERT | 11 | 0.818 | 0.321 | 0.462 |

### 3.1 Interpretation of the results

The keyword-based approach achieves the strongest overall performance, with a precision of 0.794, a recall of 0.964 and an F1-score of 0.871.

This indicates that the initial combination of sector-specific keywords and patent classifications provides a relatively effective way of identifying relevant patent families within the manually labelled sample.

TF-IDF performs less effectively, achieving an F1-score of 0.452. Although it identifies some relevant patents, its lower precision and recall indicate that lexical similarity to the focal patents is not sufficient to reproduce the performance of the keyword-based approach.

Patent SBERT based on focal patents obtains an even lower F1-score of 0.323. In this experiment, semantic similarity alone does not outperform the keyword baseline. This suggests that the selected focal patents and their abstract representations do not provide sufficiently discriminative semantic reference points for retrieving the relevant patent families in the sample.

The additional description-based SBERT experiment achieves an F1-score of 0.355, slightly higher than the focal-patent SBERT approach (0.323). Precision increases from 0.294 to 0.324, while recall increases from 0.357 to 0.393.

This indicates that using a general sector description as the semantic reference produces a modest improvement over using individual focal patents. However, the results remain considerably below those achieved by the keyword-based approach.

We also evaluate two possible combinations of keyword retrieval and focal-patent SBERT:

- **A OR B:** retrieves a patent if it is identified by either keywords or SBERT. This increases recall to 1.000, meaning that all relevant patents in the evaluation subset are retrieved. However, precision decreases to 0.491 because the combined strategy also retrieves a larger number of irrelevant patents.
- **A AND B:** retrieves a patent only if it is identified by both approaches. Precision increases to 0.818, but recall falls to 0.321. This means that the intersection produces a more selective set of candidates but misses a substantial proportion of relevant patents.

Overall, the results show that combining the keyword and focal-patent SBERT approaches does not improve the F1-score of the keyword baseline in this experiment. The description-based SBERT approach performs slightly better than focal-patent SBERT, but it does not outperform the keyword baseline either.

### 3.2 Additional semantic candidates

Finally, the code identifies patents retrieved by SBERT but not by the keyword approach, using the conditions:

```python
by_sbert & ~by_keywords
```

and, for the description-based experiment:

```python
by_sbert_description & ~by_keywords
```

The purpose of this additional step is to isolate patents that semantic similarity considers relevant candidates even though they do not contain any of the predefined sector keywords.

The execution reports:

- **23 patents** retrieved by focal-patent SBERT but not by keywords.
- **21 patents** retrieved by description-based SBERT but not by keywords.

These patents can be particularly useful for qualitative inspection: they may reveal alternative terminology, technical expressions or applications that are not adequately represented in the initial keyword list.

The manual inspection of these candidates shows that semantic similarity can identify patents related to underwater operations, marine monitoring, oceanographic observation and subsea technologies.

However, several retrieved patents concern broader marine technologies rather than underwater drones themselves. For example, some relate to floating monitoring platforms, oceanographic buoys, wave observation systems and unmanned surface vessels.

Other candidates appear more closely connected to the sector. In particular, the patent *Subsea operations support system* explicitly refers to ROVs, HROVs, AUVs and other autonomous or semi-autonomous underwater robots.

This qualitative inspection highlights an important limitation of semantic retrieval: **semantic similarity to the sector description does not necessarily imply that a patent belongs to the sector itself.**

The retrieved candidates therefore require manual assessment before being classified as relevant. The numerical count alone does not establish that these patents belong to the sector.

## 4. Preliminary conclusions

The experiments conducted so far provide four main indications:

1. **The multi-source retrieval strategy provides the initial candidate pool.** Combining CPC, IPC and keyword searches allows the identification of patent families through different technical and textual signals.

2. **The keyword-based approach is currently the strongest retrieval method.** Within the manually labelled sample, it achieves the highest F1-score and a high recall, suggesting that the selected keywords are reasonably effective in identifying relevant underwater drone patents.

3. **Semantic similarity provides an additional exploratory tool, rather than a replacement for the baseline.** Neither TF-IDF nor Patent SBERT improves the overall retrieval performance in the current experiment. The description-based SBERT approach performs slightly better than the focal-patent approach, but still does not outperform the keyword baseline.

4. **The semantic candidates can provide useful qualitative insights.** The patents retrieved exclusively by SBERT reveal alternative terminology and related technological applications that may not be captured by the initial keyword list. Nevertheless, some of these candidates concern adjacent marine technologies rather than underwater drones directly, making manual assessment necessary.

For these reasons, the initial retrieval strategy based on CPC, IPC and keywords is retained as the reference approach. The NLP experiments are documented as supplementary validation and exploration exercises, rather than being incorporated into the definitive retrieval query.

The next stage of Step 0 will focus on cleaning and preparing the retrieved patent-family dataset before proceeding to the sector analysis.

